In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
import sys
sys.path.insert(0, '..')

import torch
import numpy as np
import matplotlib.pyplot as plt
from utils import data

device = torch.device('cuda')
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# ===== Configuration =====
mouse_id     = 3
data_path    = '../data'
frozen_path  = './checkpoints/vit_frozen'
ft_path      = './checkpoints/vit_finetune'
os.makedirs(ft_path, exist_ok=True)

import os
HF_TOKEN   = os.getenv("HF_TOKEN")                                 # <-- fill in your HuggingFace token
MODEL_NAME = 'facebook/dinov3-vits16-pretrain-lvd1689m' # must match frozen notebook

# --- Token & layer config (must match frozen notebook) ---
TOKEN_TYPE     = 'patch'   # 'patch' or 'cls'
EXTRACT_LAYERS = [5]      # None = last layer; e.g. [8,9,10,11]

# --- Fine-tuning config ---
N_BLOCKS_TO_UNFREEZE = 4     # unfreeze last N of 12 blocks + token embedding layer (ViT-S)
BATCH_SIZE   = 32            # safe for RTX 3090 + ViT-S; use 16 for ViT-B
BACKBONE_LR  = 1e-5
READOUT_LR   = 1e-3
MAX_EPOCHS   = 100
PATIENCE     = 5             # early stop patience
L2_READOUT   = 1e-4

downsample_factor = 0.5        # factor to downsample ViT input (must be set in frozen notebook)

np.random.seed(1)

In [ ]:
# Load full 66×264 images WITHOUT normalization
img = data.load_images(data_path, mouse_id, file=data.img_file_name[mouse_id],
                       crop=False, normalize=False, downsample = downsample_factor)
print('raw img shape:', img.shape)

In [ ]:
# Preprocess: 66×264 → (N, 3, 32, 64), ImageNet-normalized
# Pipeline: resize 64×256 → crop left 64×128 → downsample 32×64
#           → per-image [0,1] → repeat 3ch → ImageNet normalize
from utils.vit_encoding_model import preprocess_images
img = preprocess_images(img, batch_size=2000, downsample_factor=downsample_factor)
print('preprocessed img shape:', img.shape)
print('  mean=%.4f  std=%.4f  range=[%.2f, %.2f]' % (img.mean(), img.std(), img.min(), img.max()))

In [ ]:
# Load neurons
fname = '%s_nat60k_%s.npz' % (data.db[mouse_id]['mname'], data.db[mouse_id]['datexp'])
spks, istim_train, istim_test, xpos, ypos, spks_rep_all = data.load_neurons(
    file_path=os.path.join(data_path, fname), mouse_id=mouse_id
)
n_stim, n_neurons = spks.shape

In [ ]:
itrain, ival = data.split_train_val(istim_train, train_frac=0.9)

In [ ]:
spks, spks_rep_all = data.normalize_spks(spks, spks_rep_all, itrain)

In [ ]:
ineur      = np.arange(0, n_neurons)
spks_train = torch.from_numpy(spks[itrain][:, ineur])
spks_val   = torch.from_numpy(spks[ival][:, ineur])

# Images: (N, 3, 32, 64) already from preprocess_images()
img_train = torch.from_numpy(img[istim_train][itrain]).to(device)
img_val   = torch.from_numpy(img[istim_train][ival]).to(device)
img_test  = torch.from_numpy(img[istim_test]).to(device)

print('img_train:', img_train.shape)
print('spks_train:', spks_train.shape)

In [ ]:
# ===== Rebuild the model (must match frozen notebook architecture) =====
from utils.vit_encoding_model import build_vit_encoder, make_model_name

model = build_vit_encoder(
    n_neurons        = len(ineur),
    model_name       = MODEL_NAME,
    vit_input_size   = (int(64 / downsample_factor), int(128 / downsample_factor)),   # ViT internal resize (H, W), divisible by 16
    out_spatial_size = (int(4 / downsample_factor), int(8 / downsample_factor)),     # readout resolution (half of 32×64 model input)
    extract_layers   = EXTRACT_LAYERS,
    freeze_backbone  = True,   # selectively unfrozen below
    poisson          = True,
    hf_token         = HF_TOKEN,
    device           = device,
)

In [ ]:
# ===== Load the frozen-backbone checkpoint as starting point =====
frozen_filename = make_model_name(
    data.mouse_names[mouse_id], data.exp_date[mouse_id],
    MODEL_NAME, TOKEN_TYPE, EXTRACT_LAYERS, downsample_factor
)
frozen_ckpt = os.path.join(frozen_path, frozen_filename)

if os.path.exists(frozen_ckpt):
    model.load_state_dict(torch.load(frozen_ckpt, map_location=device))
    print('Loaded frozen checkpoint:', frozen_ckpt)
else:
    print('WARNING: frozen checkpoint not found. Run vit_frozen_mouse.ipynb first.')
    print('Fine-tuning from scratch (readout will be random).')

In [ ]:
# ===== Fine-tune: unfreeze last N transformer blocks =====
from utils.vit_encoding_model import finetune_vit

ft_filename = make_model_name(
    data.mouse_names[mouse_id], data.exp_date[mouse_id],
    MODEL_NAME, TOKEN_TYPE, EXTRACT_LAYERS, downsample_factor
).replace('.pt', f'_ft{N_BLOCKS_TO_UNFREEZE}blocks.pt')
ft_ckpt = os.path.join(ft_path, ft_filename)
print('Fine-tune checkpoint:', ft_ckpt)

if not os.path.exists(ft_ckpt):
    best_state = finetune_vit(
        model,
        spks_train, spks_val,
        img_train,  img_val,
        n_blocks_to_unfreeze = N_BLOCKS_TO_UNFREEZE,
        max_epochs  = MAX_EPOCHS,
        batch_size  = BATCH_SIZE,
        backbone_lr = BACKBONE_LR,
        readout_lr  = READOUT_LR,
        l2_readout  = L2_READOUT,
        clamp       = True,
        patience    = PATIENCE,
        device      = device,
    )
    torch.save(best_state, ft_ckpt)
    print('Saved fine-tuned model:', ft_ckpt)

model.load_state_dict(torch.load(ft_ckpt, map_location=device))
print('Loaded fine-tuned model:', ft_ckpt)

In [ ]:
# ===== Evaluate fine-tuned model =====
from utils import metrics

model.eval()
ft_pred = []
with torch.no_grad():
    for k in range(0, img_test.shape[0], 64):
        pred = model(img_test[k:k+64])
        ft_pred.append(pred.cpu().numpy())
ft_pred = np.vstack(ft_pred)

ft_fev, ft_feve = metrics.feve(spks_rep_all, ft_pred)

threshold = 0.15
valid = np.where(ft_fev > threshold)[0]
print(f'Valid neurons (FEV > {threshold}): {len(valid)} / {len(ft_fev)}')
print(f'FEVE (ViT fine-tuned {N_BLOCKS_TO_UNFREEZE} blocks): {np.mean(ft_feve[ft_fev > threshold]):.4f}')

In [ ]:
# ===== Load frozen model predictions for comparison =====
# Re-build and reload frozen model
frozen_model = build_vit_encoder(
    n_neurons        = len(ineur),
    model_name       = MODEL_NAME,
    vit_input_size   = (int(64 / downsample_factor), int(128 / downsample_factor)),   # ViT internal resize (H, W), divisible by 16
    out_spatial_size = (int(4 / downsample_factor), int(8 / downsample_factor)),     # readout resolution (half of 32×64 model input)
    extract_layers   = EXTRACT_LAYERS,
    freeze_backbone  = True,
    poisson          = True,
    hf_token         = HF_TOKEN,
    device           = device,
)
if os.path.exists(frozen_ckpt):
    frozen_model.load_state_dict(torch.load(frozen_ckpt, map_location=device))
    frozen_model.eval()
    frozen_pred = []
    with torch.no_grad():
        for k in range(0, img_test.shape[0], 64):
            frozen_pred.append(frozen_model(img_test[k:k+64]).cpu().numpy())
    frozen_pred = np.vstack(frozen_pred)
    frozen_fev, frozen_feve = metrics.feve(spks_rep_all, frozen_pred)
    print(f'FEVE (ViT frozen):       {np.mean(frozen_feve[frozen_fev > threshold]):.4f}')
else:
    frozen_pred = None
    print('Frozen checkpoint not found — skipping comparison')